In [0]:
# Notebook: 05_Batch_Scoring
from databricks import feature_store
import mlflow

# --- Configuration ---
model_name = "databricks_us.default.adventureworkssalespredictor"
model_stage = "staging" # Or "Staging" / "Production" if you transitioned it
output_table_name = "adventureworks_db.predictions.sales_predictions2" # Where to save predictions

In [0]:
# --- Load Data to Score ---
# Option 1: Load fresh data from the source database
# Re-use connection logic from 01_Data_Ingestion notebook
# Filter for data that needs scoring (e.g., new orders)

# Option 2: Load data from a Delta table (e.g., daily new orders table)
# scoring_raw_df = spark.read.format("delta").load("/mnt/adventureworks/new_orders_for_scoring")
# --- Connection Configuration ---

jdbc_hostname = dbutils.secrets.get(scope="jdbc-secrets", key="db-host")
jdbc_port = dbutils.secrets.get(scope="jdbc-secrets", key="db-port")
jdbc_database = dbutils.secrets.get(scope="jdbc-secrets", key="db-database")
jdbc_user = dbutils.secrets.get(scope="jdbc-secrets", key="db-user")
jdbc_password = dbutils.secrets.get(scope="jdbc-secrets", key="db-password")

jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {
  "user": jdbc_user,
  "password": jdbc_password,
  "driver": "org.postgresql.Driver"
}

# For this example, let's just score the data we originally prepared (for demonstration)
try:
    # We only need the primary keys of the data we want to score
    # scoring_keys_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data").select("primary_key", "OrderDate") # Add timestamp if needed by features
    # OR reload from DB if not saved

    scoring_keys_df = spark.read.jdbc(
        url=jdbc_url,
        table="Sales.SalesOrderHeader", # Use actual schema.table
        properties=connection_properties
    ).select("SalesOrderID", "OrderDate").withColumnRenamed("SalesOrderID", "primary_key") # Select keys to score

    print(f"Loaded {scoring_keys_df.count()} records to score.")

except Exception as e:
     print(f"Error loading data for scoring: {e}")
     dbutils.notebook.exit("Failed to load scoring data.")


In [0]:
# --- Score using Feature Store ---
fs = feature_store.FeatureStoreClient()

# Construct the model URI based on name and stage
if model_stage == "None" or not model_stage: # Get latest version if no stage specified
     model_stage = "staging"
     model_uri = f"models:/{model_name}@{model_stage}"
else:
     model_uri = f"models:/{model_name}@{model_stage}"

print(f"Scoring using model: {model_uri}")

In [0]:
model = mlflow.pyfunc.load_model('models:/databricks_us.default.adventureworkssalespredictor@staging')
model

In [0]:
fs.score_batch

In [0]:
from pyspark.sql import functions as F

try:
    # Use score_batch - it automatically looks up features based on the model's Feature Store metadata
    predictions_df = fs.score_batch(
        model_uri='models:/databricks_us.default.adventureworkssalespredictor@staging',
        df=scoring_keys_df, # DataFrame containing primary keys (and timestamp if needed)
        result_type='double' # Or 'string', 'boolean', etc. matching model output type
    )

    print("Batch scoring completed.")
    display(predictions_df.limit(10))

    # --- Save Predictions ---
    # Add timestamp, model version used, etc. for traceability
    predictions_df = predictions_df.withColumn("prediction_timestamp", F.current_timestamp()) \
                                   .withColumn("model_version_used", F.lit(model_uri)) # Store URI or just version

    predictions_df.write.format("delta").mode("overwrite").saveAsTable(output_table_name)
    # Use append mode if adding predictions incrementally
    # predictions_df.write.format("delta").mode("append").saveAsTable(output_table_name)

    print(f"Predictions saved to Delta table: {output_table_name}")
    # dbutils.notebook.exit(output_table_name)

except Exception as e:
    print(f"Error during batch scoring or saving: {e}")
    # dbutils.notebook.exit("Batch scoring failed.")